# 02 — Live deployment (Pass 6)

Live camera detection on the Kria KV260 using a YOLOv5u xmodel compiled by
this repo's compile pipeline. Runs as text + HTML status updates (no GUI
overhead, no display thread); the final FPS is bounded by either the camera
(60 fps for the Brio at 480p MJPG) or the DPU (~16 ms per inference for
yolov5n at imgsz=320, so ~62 fps ceiling).

**Architecture**: the heavy lifting (`Preprocessor`, `decode_yolov5u`,
`ThreadedCamera`, `ModelRunner`) is factored into the `lpr_pipeline.deploy`
package. This notebook orchestrates them and renders status to the user.

**Prerequisites** — all handled by Pass 5's Kria scripts; this notebook
assumes they've already run:

| Task | Where it happens |
|---|---|
| VAI 3.5 runtime + DPU-PYNQ | `scripts/kria/01_install_vai35.sh` |
| USB autosuspend off, CPU governor, camera tuning | `scripts/kria/02_apply_tuning.sh` |
| Persistence across reboots | `scripts/kria/03_install_systemd.sh` |
| xmodel synced from laptop | `scripts/host/03_sync_to_kria.sh` |
| Jupyter launched as root with `--allow-root` | `scripts/kria/run_live.sh <variant>` |

**Stop the live loop**: click the stop button (■) in the JupyterLab toolbar,
or *Kernel → Interrupt*. Final stats are printed.


## 1. Configuration

Edit `VARIANT` to pick the model, then run the rest top-to-bottom.

If launched via `scripts/kria/run_live.sh <variant>`, the script sets
`LPR_VARIANT` and `LPR_XMODEL` in the environment and the values below
are picked up from there automatically.

`CONF_THRESH` and `IOU_THRESH` filter detections after the DPU runs.
`MAX_DETECTIONS` is a safety cap (the LPR scenes we test on have ≤5 plates,
so 300 is plenty of headroom).


In [ ]:
import os

# ────────────────────────────────────────────────────────────────────────
# Edit these to change behaviour. Other settings (camera tuning, glog,
# system tuning) are handled by Pass 5's Kria scripts and shouldn't need
# changing here.
# ────────────────────────────────────────────────────────────────────────

VARIANT        = os.environ.get("LPR_VARIANT", "yolov5n")
CONF_THRESH    = 0.30
IOU_THRESH     = 0.45
MAX_DETECTIONS = 300

# Status display refresh rates
STATUS_DT      = 1.00     # seconds between HTML status refreshes
DET_LOG_DT     = 0.40     # min seconds between detection event prints

print(f"VARIANT        = {VARIANT}")
print(f"CONF_THRESH    = {CONF_THRESH}")
print(f"IOU_THRESH     = {IOU_THRESH}")
print(f"MAX_DETECTIONS = {MAX_DETECTIONS}")


## 2. Suppress glog noise

The Vitis AI runtime prints `Check failed: r == 0 cannot set read range!` on
every model load. It's harmless (the DPU runs fine; it's a fingerprint-
verification quirk) but floods the notebook output. Setting these env vars
before any vart/xir import keeps only FATAL-level glog messages.

**Must run before the imports cell** (env vars are read at C-extension load
time, which is the first `import xir` / `import vart`).


In [ ]:
import os
os.environ['GLOG_minloglevel']     = '3'   # 0=INFO, 1=WARN, 2=ERROR, 3=FATAL
os.environ['GLOG_logtostderr']     = '0'
os.environ['GLOG_stderrthreshold'] = '3'


## 3. Imports + repo location

The repo root is normally `/home/ubuntu/KriaKv260_Model_Compiler` on the Kria.
We add it to `sys.path` so `lpr_pipeline.deploy.*` is importable. If the env
var `REPO_ROOT` is set (which `run_live.sh` does), use that; otherwise auto-
detect.


In [ ]:
import sys
from pathlib import Path

# Auto-detect repo location. Priority:
#   1. REPO_ROOT env var (set by run_live.sh)
#   2. Standard Kria location
#   3. Walk up from CWD (notebook is usually opened from notebooks/)
candidates = [
    os.environ.get("REPO_ROOT"),
    "/home/ubuntu/KriaKv260_Model_Compiler",
    str(Path.cwd().parent),
    str(Path.cwd()),
]
for cand in candidates:
    if cand and (Path(cand) / "lpr_pipeline").is_dir():
        REPO_ROOT = cand
        if cand not in sys.path:
            sys.path.insert(0, cand)
        break
else:
    raise RuntimeError(
        "Could not locate lpr_pipeline. Set REPO_ROOT env var or run via "
        "scripts/kria/run_live.sh which sets it for you."
    )

print(f"REPO_ROOT = {REPO_ROOT}")

# Pull the spec from the shared registry; build the xmodel path from VARIANT.
from lpr_pipeline.shared.models import get_spec

spec = get_spec(VARIANT)
XMODEL = os.environ.get(
    "LPR_XMODEL",
    f"/home/ubuntu/xmodels_vai35/{VARIANT}/{VARIANT}_kv260.xmodel",
)

print(f"spec   = family={spec.family} imgsz={spec.imgsz} "
      f"nc={spec.nc} reg_max={spec.reg_max}")
print(f"xmodel = {XMODEL}")

if not Path(XMODEL).exists():
    raise FileNotFoundError(
        f"xmodel missing: {XMODEL}\n"
        f"From your laptop, sync it:\n"
        f"  bash scripts/host/03_sync_to_kria.sh ubuntu@<kria-ip> {VARIANT}"
    )

# Other imports — after the env vars + sys.path are set.
import time
import threading
import numpy as np
import cv2
from IPython.display import display, HTML
from pynq_dpu import DpuOverlay

from lpr_pipeline.deploy import ModelRunner, ThreadedCamera

print(f"OpenCV {cv2.__version__}")


## 4. Load DPU overlay (program the FPGA)

The DPU bitstream is loaded once. After this cell runs, the FPGA fabric is
configured and we can hot-swap xmodels onto it via `overlay.load_model()`
without reprogramming the hardware.

This takes 3-5 seconds on first run. Subsequent runs (without restarting
the kernel) skip the bitstream programming if it's already loaded.


In [ ]:
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded — FPGA programmed and ready for xmodels.")


## 5. Build the ModelRunner and warm it up

`ModelRunner` ties together the preprocessor (letterbox + normalize), the
DPU runner (PYNQ-DPU's `overlay.runner`), and the decoder (`decode_yolov5u`).

Warmup runs 5 inferences on random input. The first is always slower due
to JIT / buffer allocation / cache warming. By the 3rd or 4th, timing
stabilizes at the steady-state.


In [ ]:
runner = ModelRunner(spec, XMODEL, overlay)

print(f"\nModelRunner built:")
print(f"  input  dims = {runner.input_dims}")
for i, d in enumerate(runner.output_dims):
    print(f"  output[{i}] = {d}")

print(f"\nWarming up ({5} runs):")
warmup_times = runner.warmup(n=5, print_each=True)
print(f"\nSteady-state ≈ {min(warmup_times[2:]):.2f} ms (best of last 3)")


## 6. Pure inference benchmark (DPU + decode ceiling)

Measures the maximum inference rate when fed an in-memory frame — no camera,
no display. This is the *thesis ceiling* number: how fast the inference
pipeline can run if everything else were free.

The reportable number is `mean_ms`; throughput is just `1000 / mean_ms`.
Per-stage timings tell you where the time is going (almost always:
`dpu >> decode >> preprocess`).


In [ ]:
def benchmark_pure(n=200, warmup=20):
    # Pure-inference benchmark with per-stage breakdown.
    frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)
    for _ in range(warmup):
        runner.infer(frame)

    total_times = np.empty(n, dtype=np.float64)
    pre_times   = np.empty(n, dtype=np.float64)
    dpu_times   = np.empty(n, dtype=np.float64)
    dec_times   = np.empty(n, dtype=np.float64)

    for i in range(n):
        t0 = time.perf_counter()
        _, t = runner.infer(frame)
        total_times[i] = (time.perf_counter() - t0) * 1000
        pre_times[i]   = t["preprocess"]
        dpu_times[i]   = t["dpu"]
        dec_times[i]   = t["decode"]

    def stats(arr, label):
        print(f"  {label:>10s}  "
              f"mean={arr.mean():6.2f}  "
              f"p50={np.percentile(arr,50):6.2f}  "
              f"p95={np.percentile(arr,95):6.2f}  "
              f"p99={np.percentile(arr,99):6.2f}")

    print(f"=== {VARIANT} pure inference (n={n}) — timings in ms ===")
    stats(total_times, "total")
    stats(pre_times,   "preprocess")
    stats(dpu_times,   "dpu")
    stats(dec_times,   "decode")
    print(f"\n  → throughput = {1000 / total_times.mean():6.1f} fps  "
          f"(p95: {1000 / np.percentile(total_times, 95):.1f} fps)")
    return total_times, pre_times, dpu_times, dec_times

bench_total, bench_pre, bench_dpu, bench_dec = benchmark_pure()


## 7. Live loop — camera → DPU → status display

Threaded camera (BUFFERSIZE=4, MJPG, 60 fps) feeds the runner; the status
line refreshes once per second; detection events stream below (throttled
to once per 0.4 s so a stationary plate doesn't spam).

**Stop**: stop button (■) in the JupyterLab toolbar, or
*Kernel → Interrupt*. Final stats are printed afterward.

The HTML status block reports:

| Field | Meaning |
|---|---|
| `inf_fps` | End-to-end inferences per second (camera + preprocess + DPU + decode) |
| `cam_fps` | Unique frames the camera produced — caps `inf_fps` from above |
| `pre / dpu / dec` | Per-stage EMA latencies in ms |
| `theoretical_max` | `1000 / (pre+dpu+dec)` — pipeline's ceiling if camera were infinite |
| `detections` | Total detections across all frames |
| `hit_rate` | % of inferred frames that found at least one detection |


In [ ]:
# Make sure no leftover camera handle from a previous run leaves /dev/video0 busy
try:
    cam.close()
    print("(closed prior camera)")
except (NameError, AttributeError):
    pass

cam = ThreadedCamera()
print(f"  camera: {cam.actual_width}x{cam.actual_height} "
      f"@ {cam.actual_fps:.0f} fps  BUFFERSIZE=4")

status = display(
    HTML("<pre style='font-family:monospace'>starting...</pre>"),
    display_id=True,
)

# Counters
n_inf, n_dets_total, n_frames_with_d = 0, 0, 0
unique_ids = set()
# EMAs (exponentially-weighted moving averages) for per-stage timings.
# Alpha=0.1 → 90% old + 10% new; smooth enough to read at 1 Hz update rate.
ema_pre, ema_dpu, ema_dec = 0.0, 0.0, 0.0

t_start         = time.perf_counter()
last_status_t   = t_start
last_dets_log_t = 0.0

print(f"\n[ live {VARIANT}  •  press stop (■) to end ]\n")

try:
    while True:
        frame, fid = cam.read_new()
        if frame is None:
            time.sleep(0.001)
            continue
        unique_ids.add(fid)

        dets, t = runner.infer(frame, conf=CONF_THRESH, iou=IOU_THRESH,
                                max_detections=MAX_DETECTIONS)
        n_inf += 1

        # Update EMAs. First call seeds them so we don't start at 0.
        if ema_pre:
            ema_pre = 0.9 * ema_pre + 0.1 * t["preprocess"]
            ema_dpu = 0.9 * ema_dpu + 0.1 * t["dpu"]
            ema_dec = 0.9 * ema_dec + 0.1 * t["decode"]
        else:
            ema_pre, ema_dpu, ema_dec = t["preprocess"], t["dpu"], t["decode"]

        if dets:
            n_dets_total    += len(dets)
            n_frames_with_d += 1

        now = time.perf_counter()

        # Detection events (throttled).
        if dets and (now - last_dets_log_t) >= DET_LOG_DT:
            last_dets_log_t = now
            for x1, y1, x2, y2, conf, cls_idx in dets:
                # Class names: LPR is 1-class. For multi-class we'd read from
                # spec or a separate names list.
                cls_name = "plate" if spec.nc == 1 else str(cls_idx)
                print(f"  ▶ {cls_name}  "
                      f"({int(x1):3d},{int(y1):3d}) → ({int(x2):3d},{int(y2):3d})  "
                      f"conf={conf:.3f}")

        # Status refresh.
        if (now - last_status_t) >= STATUS_DT:
            last_status_t = now
            elapsed = now - t_start
            inf_fps = n_inf / elapsed
            cam_fps = len(unique_ids) / elapsed
            hit_pct = n_frames_with_d / n_inf * 100 if n_inf else 0
            ema_total = ema_pre + ema_dpu + ema_dec
            html = (
                "<pre style='font-family:monospace;font-size:13px;"
                "background:#1e1e1e;color:#d4d4d4;padding:8px;"
                "border-radius:4px'>"
                f"<b style='color:#4ec9b0'>{VARIANT}</b>  "
                f"elapsed={elapsed:6.1f}s  frames={n_inf:6d}\n"
                f"<b style='color:#dcdcaa'>inf_fps</b>={inf_fps:5.1f}  "
                f"<b style='color:#dcdcaa'>cam_fps</b>={cam_fps:5.1f}  "
                f"<b style='color:#dcdcaa'>theoretical_max</b>="
                f"{1000/ema_total if ema_total else 0:5.1f} fps\n"
                f"<b style='color:#9cdcfe'>pre</b>={ema_pre:4.2f}  "
                f"<b style='color:#9cdcfe'>dpu</b>={ema_dpu:5.2f}  "
                f"<b style='color:#9cdcfe'>dec</b>={ema_dec:4.2f}  "
                f"(<b style='color:#9cdcfe'>total</b>={ema_total:5.2f} ms)\n"
                f"<b style='color:#c586c0'>detections</b>={n_dets_total}  "
                f"<b style='color:#c586c0'>hit_rate</b>={hit_pct:5.1f}%"
                "</pre>"
            )
            status.update(HTML(html))

except KeyboardInterrupt:
    print("\n[ stopped by user ]")
finally:
    cam.close()
    elapsed = time.perf_counter() - t_start
    inf_fps = n_inf / elapsed if elapsed > 0 else 0
    cam_fps = len(unique_ids) / elapsed if elapsed > 0 else 0
    hit_pct = n_frames_with_d / n_inf * 100 if n_inf else 0
    ema_total = ema_pre + ema_dpu + ema_dec

    print(f"\n=== final stats — {VARIANT} ===")
    print(f"  total time            : {elapsed:8.2f} s")
    print(f"  frames inferred       : {n_inf:8d}")
    print(f"  unique camera frames  : {len(unique_ids):8d}")
    print(f"  inference fps         : {inf_fps:8.2f}")
    print(f"  camera fps            : {cam_fps:8.2f}")
    print(f"  avg preprocess (ms)   : {ema_pre:8.2f}")
    print(f"  avg dpu        (ms)   : {ema_dpu:8.2f}")
    print(f"  avg decode     (ms)   : {ema_dec:8.2f}")
    print(f"  avg total      (ms)   : {ema_total:8.2f}")
    print(f"  theoretical max       : {1000/ema_total if ema_total else 0:8.2f} fps")
    print(f"  total detections      : {n_dets_total:8d}")
    print(f"  frames with object    : {n_frames_with_d:8d}")
    print(f"  hit rate              : {hit_pct:8.2f} %")


## 8. Notes

### Switching models

Change `VARIANT` in cell 1 and **restart the kernel** (Kernel → Restart),
then run all cells again. The DPU configuration changes per xmodel and
the runner state needs a clean slate.

When launching via `scripts/kria/run_live.sh <variant>`, the env var
`LPR_VARIANT` is set automatically and cell 1's default reads it — so the
notebook picks up the right variant without any manual edit. (You still
need to restart the kernel when switching variants if the kernel was
already running.)

### Reading the numbers

| Number | What it means | Bound by |
|---|---|---|
| `inf_fps` | End-to-end throughput | camera or DPU, whichever is slower |
| `cam_fps` | Camera's unique-frame rate | Brio @ 60 |
| `dpu` | DPU compute time per inference | model size + DPU config |
| `pre` + `dec` | CPU work per inference | the four Cortex-A53s on KV260 |
| `theoretical_max` | Pipeline ceiling | `1000 / (pre + dpu + dec)` |

If `inf_fps ≈ cam_fps ≈ 60` on yolov5n → pipeline is camera-bound; the
DPU has spare capacity. That's the answer for the live demo.

The pure benchmark in cell 6 gives the DPU-only inference latency, which
is the right number for the *inference-latency* claim in the thesis
(distinct from end-to-end throughput).

### Where stuff lives

| Thing | File |
|---|---|
| `Preprocessor`, `unletterbox` | `lpr_pipeline/deploy/preprocess.py` |
| `decode_yolov5u` | `lpr_pipeline/deploy/decoders.py` |
| `ThreadedCamera` | `lpr_pipeline/deploy/camera.py` |
| `ModelRunner` (this notebook's main driver) | `lpr_pipeline/deploy/runner.py` |
| Model specs (imgsz, nc, reg_max) | `lpr_pipeline/shared/models.py` |
| This notebook | `notebooks/02_deploy_live.ipynb` |

### What this notebook deliberately *doesn't* do

- **System tuning** (USB autosuspend, CPU governor, v4l2 settings):
  handled by `scripts/kria/02_apply_tuning.sh`, persisted by the
  systemd unit `kriakv260-tuning.service`.
- **Evaluation / mAP**: deferred to a future pass; see
  `notebooks/03_deploy_eval.ipynb` (TBD).
- **YOLOX**: spec exists, decoder doesn't yet. yolov5n and yolov5s only.
